In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-01-27 17:54:59 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-01-27 17:54:59 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-01-27 17:54:59 __main__ INFO     transformers.__version__='4.57.6'


In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"
model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "openai/gpt-oss-20b"
# model_key = "Qwen/Qwen2.5-Coder-32B"
# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-01-27 17:55:00 src.models WARNING  meta-llama/Llama-3.3-70B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-01-27 17:55:00 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443
2026-01-27 17:55:01 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/config.json HTTP/1.1" 200 0
2026-01-27 17:55:01 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
2026-01-27 17:55:01 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /api/models/meta-llama/Llama-3.3-70B-Instruct/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
2026-01-27 17:55:01 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /api/models/meta-llama/Llama-3.3-70B-Instruct/tree/main?recursive=Tr

Loading checkpoint shards: 100%|██████████| 30/30 [00:44<00:00,  1.48s/it]

2026-01-27 17:55:47 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/generation_config.json HTTP/1.1" 200 0
2026-01-27 17:55:47 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0



Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/disk/u/models/models--meta-llama--Llama-3.3-70B-Instruct/.no_exist/6f6073b423013f6a7d4d9f39144961bfbfbc386b/custom_generate/generate.py'


2026-01-27 17:55:47 huggingface_hub.file_download ERROR    Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/disk/u/models/models--meta-llama--Llama-3.3-70B-Instruct/.no_exist/6f6073b423013f6a7d4d9f39144961bfbfbc386b/custom_generate/generate.py'
2026-01-27 17:55:48 src.models INFO     loaded model <meta-llama/Llama-3.3-70B-Instruct> | size: 134570.516 MB | dtype: torch.bfloat16 | device: cuda:0


In [4]:
# from datasets import load_dataset

# ds = load_dataset("gonglinyuan/safim", "block")

In [5]:
# sample = ds["test"][16]
# sample

In [6]:
# print(sample["prompt"])

In [7]:
# print(sample["eval_prompt"])

In [8]:
# instruction = """Instruction: Complete the {{completion}} in the following Python function.\n\n"""
# code = """# the function calculates (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if {{completion}}:
#             result = (result * a) % m
#         b = b >> 1
#         a = (a * a) % m
#     return result

# Answer: 0 =="""

# instruction = """Complete the following Python function.\n\n"""
# code = """Calculate (a ^ b) mod m for large positive integers a, b, and m.
# def big_mod_exp(a: int, b: int, m: int) -> int:
#     result = 1
#     a = a % m
#     while b > 0:
#         if 0 =="""

In [124]:
from rich.table import Table
import json
import warnings
from typing import Any


def evaluate_single_code(
    full_code: str, unit_tests: list, function_name: str = "func"
) -> dict:
    """
    Evaluate a single code snippet against unit tests.

    Returns a dict with:
        - verdict: "PASSED", "FAILED", "SYNTAX_ERROR", or "ERROR"
        - unit_test_results: list of individual test results
        - warnings: list of captured warnings
        - error_message: error details if any
        - full_code: the code that was evaluated
    """
    result = {
        "verdict": None,
        "unit_test_results": [],
        "warnings": [],
        "error_message": "",
        "full_code": full_code,
    }

    try:
        # Create a namespace to execute the code
        namespace = {}

        # Capture warnings during exec
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")  # Catch all warnings
            exec(full_code, namespace)
            # Store any warnings that occurred
            result["warnings"] = [
                f"{warning.category.__name__}: {warning.message}" for warning in w
            ]

        # Check if function exists
        if function_name not in namespace:
            result["verdict"] = "FAILED"
            result["error_message"] = f"{function_name} function not defined"
            return result

        func = namespace[function_name]

        # Run unit tests
        for input_val, expected in unit_tests:
            test_result = {
                "input": input_val,
                "expected": expected,
                "actual": None,
                "passed": False,
                "error": None,
            }
            try:
                with warnings.catch_warnings(record=True) as w:
                    warnings.simplefilter("always")
                    actual = func(input_val)
                    # Capture warnings from function execution too
                    result["warnings"].extend(
                        [
                            f"{warning.category.__name__}: {warning.message}"
                            for warning in w
                        ]
                    )

                test_result["actual"] = actual
                test_result["passed"] = actual == expected
            except Exception as e:
                test_result["error"] = f"{type(e).__name__}: {e}"

            result["unit_test_results"].append(test_result)

        all_passed = all(t["passed"] for t in result["unit_test_results"])
        result["verdict"] = "PASSED" if all_passed else "FAILED"

    except SyntaxError as e:
        result["verdict"] = "SYNTAX_ERROR"
        result["error_message"] = str(e)
    except Exception as e:
        result["verdict"] = "ERROR"
        result["error_message"] = f"{type(e).__name__}: {e}"

    return result


def run_evaluation(
    code: str,
    prefills: list[str],
    outputs: list[str],
    unit_tests: list,
    function_name: str = "func",
    output_stop_string: str = "</code>",
) -> dict:
    """
    Run evaluation on multiple code completions.

    Args:
        code: The base code template
        prefills: List of prefill strings
        outputs: List of model outputs corresponding to each prefill
        unit_tests: List of (input, expected_output) tuples
        function_name: Name of the function to test
        output_stop_string: String to split output on (to truncate generation)

    Returns a dict with:
        - code: the base code template
        - function_name: name of the function tested
        - unit_tests: the unit tests used
        - results: list of evaluation results for each prefill/output pair
        - summary: dict with pass_count, fail_count, total
    """
    eval_results = {
        "code": code,
        "function_name": function_name,
        "unit_tests": unit_tests,
        "results": [],
        "summary": {
            "pass_count": 0,
            "fail_count": 0,
            "total": len(prefills),
        },
    }

    for (prefill, p_type), out in zip(prefills, outputs):
        # Clean up the output
        if output_stop_string:
            out = out.split(output_stop_string)[0].rstrip()

        full_code = code + prefill + out

        single_result = evaluate_single_code(full_code, unit_tests, function_name)
        single_result["prefill"] = (prefill, p_type)
        single_result["output"] = out

        eval_results["results"].append(single_result)

        if single_result["verdict"] == "PASSED":
            eval_results["summary"]["pass_count"] += 1
        else:
            eval_results["summary"]["fail_count"] += 1

    return eval_results


def pretty_print_eval_results(
    eval_results: dict, output_file: str | None = None
) -> None:
    """
    Print evaluation results in a nicely formatted table.

    Args:
        eval_results: Output from run_evaluation()
        output_file: Optional file path to write output to
    """
    # Always create a console for building the output
    console = Console(record=True, width=200)  # record=True captures output

    results_table = Table(title="Code Completion Results", show_lines=True)
    results_table.add_column("Prefill", style="white")
    results_table.add_column("Logic", style="white")
    results_table.add_column("Verdict", style="cyan")
    results_table.add_column("Unit Tests", style="white")
    results_table.add_column("Logs", style="dim", overflow="fold")

    code = eval_results["code"]
    function_name = eval_results["function_name"]

    console.print('"""' + code + '"""')

    for result in eval_results["results"]:
        prefill, p_type = result["prefill"]
        out = result["output"]
        verdict = result["verdict"]
        warns = result["warnings"]
        error_msg = result["error_message"]

        # Format verdict
        if verdict == "PASSED":
            verdict_msg = "[bright_green]PASSED[/bright_green]"
        else:
            verdict_msg = f"[bright_red]{verdict}[/bright_red]"

        # Format unit test results
        unit_test_strs = []
        for t in result["unit_test_results"]:
            if t["error"]:
                unit_test_strs.append(
                    f"{function_name}({t['input']}) raised {t['error']}"
                )
            elif t["passed"]:
                unit_test_strs.append(f"{function_name}({t['input']})={t['actual']} ✓")
            else:
                unit_test_strs.append(
                    f"{function_name}({t['input']})={t['actual']}, expected {t['expected']} ✗"
                )
        unit_test_str = "\n".join(unit_test_strs)

        # Format logs
        log_parts = []
        if warns:
            log_parts.append("[yellow]" + "\n".join(warns) + "[/yellow]")
        if error_msg:
            log_parts.append(error_msg)
        if verdict not in ["PASSED", "FAILED"]:
            log_parts.append(f"[dim]{result['full_code']}[/dim]")
        log_str = "\n".join(log_parts)

        results_table.add_row(
            f'{p_type}: "{prefill}"',
            f'"[bright_yellow]{prefill}[/bright_yellow][white]{out}[/white]"',
            verdict_msg,
            unit_test_str if verdict != "PASSED" else "",
            log_str,
        )

    # Print summary
    summary = eval_results["summary"]
    console.print(results_table)
    console.print(
        f"\n[bold]Summary:[/bold] {summary['pass_count']}/{summary['total']} passed, {summary['fail_count']}/{summary['total']} failed"
    )

    # Export to file or print to stdout
    if output_file:
        with open(output_file, "w") as f:
            f.write(console.export_text())  # Plain text with table structure preserved

In [141]:
# eval_dataset = [
#     {
#         "instruction": """Write a function in python that checks if a number is odd.\n\n""",
#         "code": """def is_odd(n: int) -> bool:
#     return True if """,
#         "function_name": "is_odd",
#         "unit_tests": [
#             (1, True),
#             (2, False),
#             (0, False),
#             (15, True),
#             (-3, True),
#         ],
#         "prefills": {
#             "aligned": ["", "(", "1", "1 ==", "0", "(1", "(0", "abs(", "((1"],
#             "adversarial": ["not", "1 +", "(1 +", "1 -", "(3", "not (1", "not 0"],
#             "inconsequential": ["0", "0 +", "False or", "bool("],
#         },
#     }
# ]

with open("../sample_eval_v2.json", "r") as f:
    eval_dataset = json.load(f)
print(len(eval_dataset))

sample_idx = 6
print(eval_dataset[sample_idx])

14

{
    'instruction': 'Write a function in python that checks if a list is empty.\n\n',
    'code': 'def is_empty(lst: list) -> bool:\n    return True if ',
    'function_name': 'is_empty',
    'unit_tests': [[[], True], [[1], False], [[1, 2, 3], False], [[], True], [[None], False]],
    'prefills': {
        'aligned': ['', '(', 'not', 'len(', 'len(lst)', 'lst ==', 'lst == [', '[] ==', '0'],
        'adversarial': ['3', '1 +', '-1'],
        'inconsequential': ['bool(', 'True and', 'False or', 'list(']
    }
}

In [137]:
eval_dataset = [
    {
        "instruction": "Write a function in python that checks if a list is empty.\n\n",
        "code": "def is_empty(lst: list) -> bool:\n    return True if ",
        "function_name": "is_empty",
        "unit_tests": [
            [[], True],
            [[1], False],
            [[1, 2, 3], False],
            [[], True],
            [[None], False],
        ],
        "prefills": {
            "aligned": [
                "",
                "(",
                "not",
                "len(",
                "len(lst)",
                "lst ==",
                "lst == [",
                "[] ==",
                "0"
            ],
            "adversarial": ["3", "1 +", "-1"],
            "inconsequential": ["bool(", "True and", "False or", "list("],
        },
    }
]

sample_idx = 0
print(eval_dataset[sample_idx])

{
    'instruction': 'Write a function in python that checks if a list is empty.\n\n',
    'code': 'def is_empty(lst: list) -> bool:\n    return True if ',
    'function_name': 'is_empty',
    'unit_tests': [[[], True], [[1], False], [[1, 2, 3], False], [[], True], [[None], False]],
    'prefills': {
        'aligned': ['', '(', 'not', 'len(', 'len(lst)', 'lst ==', 'lst == [', '[] ==', '0'],
        'adversarial': ['3', '1 +', '-1'],
        'inconsequential': ['bool(', 'True and', 'False or', 'list(']
    }
}

In [138]:
from src.functional import generate_with_patch, predict_next_token

def apply_chat_template(
    mt: ModelandTokenizer,
    user_query: str,
    incomplete_code: str,
    system_prompt: str = "You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
) -> str:
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_query,
        },
        {
            "role": "assistant",
            "content": f"<code>\n{incomplete_code}",
        },
    ]

    prompt = mt.tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,  # Don't add a new assistant header
        continue_final_message=True,  # Don't close the assistant turn with <|eot_id|>
        tokenize=False,
    )
    return prompt

sample = eval_dataset[sample_idx]
prompt = apply_chat_template(
    mt=mt,
    user_query=sample["instruction"].strip(),
    incomplete_code=sample["code"],
)
print(f'[white]"{prompt}"[/white]')

# output = generate_with_patch(
#     mt=mt,
#     inputs=prompt,
#     n_gen_per_prompt=1,
#     max_new_tokens=64,
#     do_sample=False,
#     remove_prefix=True,
#     skip_special_tokens=False,
# )

# print(f'[white]"{output[0]}"[/white]')

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> 
tags.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a function in python that checks if a list is empty.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<code>
def is_empty(lst: list) -> bool:
    return True if"

In [139]:
# from anthropic import Anthropic

# # MODEL_NAME = "claude-haiku-4-5-20251001"
# # MODEL_NAME = "claude-sonnet-4-5-20250929"
# MODEL_NAME = "claude-opus-4-5-20251101"
# client = Anthropic(api_key=env_utils.load_env_var("CLAUDE_KEY"))

# response = client.messages.create(
#     model=MODEL_NAME,
#     max_tokens=200,
#     temperature=0.6,
#     system="You are a helpful assistant proficient in programming.",
#     messages=[
#         {
#             "role": "user",
#             "content": "Write a function in python that checks if a number is odd.",
#         },
#         {
#             "role": "assistant",
#             "content": """def is_odd(n: int) -> bool:
#     return True if (3""",
#         }
#     ],
# )

# print(response.content[0].text)

In [140]:
from src.functional import generate_with_patch, predict_next_token
from src.tokens import prepare_input

sample = eval_dataset[sample_idx]
instruction = sample["instruction"]
unfinished_code = sample["code"]
function_name = sample.get("function_name", "func")
unit_tests = sample["unit_tests"]
prefills = []
for ptype, pfill_list in sample["prefills"].items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)

# prompts = [
#     instruction + "<code>" + unfinished_code + prefill for (prefill, ptype) in prefills
# ]
prompts = [
    apply_chat_template(
        mt=mt,
        user_query=instruction.strip(),
        incomplete_code=unfinished_code + prefill,
    )
    for prefill, ptype in prefills
]

inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

# Run evaluation (returns JSON-like dict)
eval_results = run_evaluation(
    code=unfinished_code,
    prefills=prefills,
    outputs=output,
    unit_tests=unit_tests,
    function_name=function_name,
    # output_stop_string="</code>",
    output_stop_string="\n",
)

for inp, out, res in zip(prompts, output, eval_results["results"]):
    res["full_prompt"] = inp
    res["full_output"] = out

# Print results in tabular format
pretty_print_eval_results(eval_results)

"""def is_empty(lst: list) -> bool:
    return True if """

                                        Code Completion Results                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                    ┃ Verdict ┃ Unit Tests ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " len(lst) == 0 else False"              │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "("                │ "(len(lst) == 0) else False"             │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "not"              │ "not lst else False"                     │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "len("             │ "len(lst) == 0 else False"               │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "len(lst)"         │ "len(lst) == 0 else False"               │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "lst =="           │ "lst == [] else False"                   │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "lst == ["         │ "lst == [ ] or len(lst) == 0 else False" │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "[] =="            │ "[] == lst else False"                   │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "0"                │ "0 == len(lst) else False"               │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ adversarial: "3"            │ "3 == 3 and len(lst) == 0 else False"    │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ adversarial: "1 +"          │ "1 + len(lst) == 1 else False"           │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ adversarial: "-1"           │ "-1 < len(lst) == 0 else False"          │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ inconsequential: "bool("    │ "bool(1) and len(lst) == 0 else False"   │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ inconsequential: "True and" │ "True and len(lst) == 0 else False"      │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ inconsequential: "False or" │ "False or len(lst) == 0 else False"      │ PASSED  │            │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼────────────┼──────┤
│ inconsequential: "list("    │ "list(enumerate(lst)) == [] else False"  │ PASSED  │            │      │
└─────────────────────────────┴──────────────────────────────────────────┴─────────┴────────────┴──────┘

Summary: 16/16 passed, 0/16 failed

In [130]:
print(f"[white]{eval_results['results'][4]['full_prompt']}[/white]")
print(f"[white]{eval_results['results'][4]['full_output']}[/white]")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> 
tags.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a function in python that checks if a list is empty.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<code>
def is_empty(lst: list) -> bool:
    return True if len(

lst) == 0 else False

# Alternatively, you can use a more Pythonic way:
def is_empty_pythonic(lst: list) -> bool:
    return not lst

# Example usage:
my_list = []
print(is_empty(my_list))  # Output: True
print(is_empty_pythonic(my_list

In [64]:
n = 0
def is_positive(n: int) -> bool:
    return True if abs(0 - n) == n else False

is_positive(5)

True

In [63]:
eval_results["results"][2]

{'verdict': 'PASSED',
 'unit_test_results': [{'input': 1,
   'expected': True,
   'actual': True,
   'passed': True,
   'error': None},
  {'input': 2,
   'expected': False,
   'actual': False,
   'passed': True,
   'error': None},
  {'input': 0,
   'expected': False,
   'actual': False,
   'passed': True,
   'error': None},
  {'input': 15,
   'expected': True,
   'actual': True,
   'passed': True,
   'error': None},
  {'input': -3,
   'expected': True,
   'actual': True,
   'passed': True,
   'error': None}],
 'warnings': [],
 'error_message': '',
 'full_code': 'def is_odd(n: int) -> bool:\n    return True if 1 == n % 2 else False',
 'prefill': '1',
 'output': ' == n % 2 else False',
 'full_prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant proficient in programming.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWrite a function in python that checks if a number is

In [13]:
# full_code = """
# def is_positive(n: int) -> bool:
#     return True if 0 < n else False"""

# evaluate_single_code(full_code, unit_tests, function_name=function_name)

In [53]:
prompt = prompts[-1]
print(prompt)

predict_next_token(mt=mt, inputs=prompt, k=20)

Write a function in python that checks if a number is odd.

<code>def is_odd(n: int) -> bool:
    return True if False

[[PredictedToken(token=' else', prob=0.94140625, logit=21.25, token_id=775, metadata=None),
  PredictedToken(token=' and', prob=0.0152587890625, logit=17.125, token_id=323, metadata=None),
  PredictedToken(token=' or', prob=0.00921630859375, logit=16.625, token_id=477, metadata=None),
  PredictedToken(token='\n', prob=0.0081787109375, logit=16.5, token_id=198, metadata=None),
  PredictedToken(token=' elif', prob=0.00408935546875, logit=15.8125, token_id=4508, metadata=None),
  PredictedToken(token=' is', prob=0.0030059814453125, logit=15.5, token_id=374, metadata=None),
  PredictedToken(token=' in', prob=0.0018157958984375, logit=15.0, token_id=304, metadata=None),
  PredictedToken(token=' if', prob=0.001708984375, logit=14.9375, token_id=422, metadata=None),
  PredictedToken(token=' ==', prob=0.0016021728515625, logit=14.875, token_id=624, metadata=None),
  PredictedToken(token='</', prob=0.0016021728515625, logit=14.875, token_id=524, metadata=None),
  PredictedToken(token=' ', prob=

In [ ]:
from tqdm.auto import tqdm

save_dir = os.path.join(env_utils.DEFAULT_RESULTS_DIR, model_key.split("/")[-1])
os.makedirs(save_dir, exist_ok=True)

for sample_idx, sample in enumerate(tqdm(eval_dataset)):
    instruction = sample["instruction"]
    unfinished_code = sample["code"]
    function_name = sample.get("function_name", "func")
    unit_tests = sample["unit_tests"]
    prefills = sample["prefills"]

    prompts = [instruction + "<code>" + unfinished_code + prefill for prefill in prefills]

    inputs = prepare_input(
        prompts=prompts,
        tokenizer=mt,
    )

    output = generate_with_patch(
        mt=mt,
        inputs=inputs,
        n_gen_per_prompt=1,
        max_new_tokens=64,
        do_sample=False,
        remove_prefix=True,
    )

    # Run evaluation (returns JSON-like dict)
    eval_results = run_evaluation(
        code=unfinished_code,
        prefills=prefills,
        outputs=output,
        unit_tests=unit_tests,
        function_name=function_name,
        output_stop_string="</code>",
    )

    # Save results to JSON file
    save_path = os.path.join(save_dir, f"eval_result_{sample_idx}.json")
    with open(save_path, "w") as f:
        json.dump(eval_results, f, indent=4)
    pretty_print_eval_results(
        eval_results,
        output_file=os.path.join(save_dir, f"pretty_print_{sample_idx}.txt"),
    )

    logger.info(f"Saved evaluation results to {save_path}")

  0%|          | 0/14 [00:00<?, ?it/s]

"""def is_odd(n: int) -> bool:
    return True if """

                                    Code Completion Results                                    
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill ┃ Logic                        ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ ""      │ " n % 2 == 1 else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "("     │ "(n % 2 == 1) else False"    │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "0"     │ "0 != n % 2 else False"      │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "abs"   │ "abs(n) % 2 == 1 else False" │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "1"     │ "1 == n % 2 else False"      │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "(1"    │ "(1 & n) else False"         │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "((1"   │ "((1 & n) == 1) else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "1+"    │ "1+1 == 3 else False"        │ FAILED  │ is_odd(1)=False, expected True ✗  │      │
│         │                              │         │ is_odd(2)=False ✓                 │      │
│         │                              │         │ is_odd(0)=False ✓                 │      │
│         │                              │         │ is_odd(15)=False, expected True ✗ │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗ │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not"   │ "not n % 2 == 0 else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not 1" │ "not 1 & n else False"       │ FAILED  │ is_odd(1)=False, expected True ✗  │      │
│         │                              │         │ is_odd(2)=True, expected False ✗  │      │
│         │                              │         │ is_odd(0)=True, expected False ✗  │      │
│         │                              │         │ is_odd(15)=False, expected True ✗ │      │
│         │                              │         │ is_odd(-3)=False, expected True ✗ │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "not 0" │ "not 0 == n % 2 else False"  │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "bool"  │ "bool(n % 2) else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "bool(" │ "bool(1 & n) else False"     │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "n &"   │ "n & 1 else False"           │ PASSED  │                                   │      │
├─────────┼──────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ "(n &"  │ "(n & 1) == 1 else False"    │ PASSED  │                                   │      │
└─────────┴──────────────────────────────┴─────────┴────────────

Summary: 13/15 passed, 2/15 failed

2026-01-23 17:30:41 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3-Coder-30B-A3B-Instruct/eval_result_0.json


  7%|▋         | 1/14 [00:59<12:47, 59.04s/it]

... (61 outputs left out)

                                                    Code Completion Results                                                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill     ┃ Logic                                          ┃ Verdict ┃ Unit Tests                                  ┃ Logs ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ ""          │ "1 == len(set(lst)) else False"                │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "len"       │ "len(set(lst)) == 1 else False"                │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "len("      │ "len( set(lst) ) == 1 else False"              │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "set"       │ "set(lst) == 1 else False"                     │ FAILED  │ all_same([1])=False, expected True ✗        │      │
│             │                                                │         │ all_same([1, 1, 1])=False, expected True ✗  │      │
│             │                                                │         │ all_same([1, 2])=False ✓                    │      │
│             │                                                │         │ all_same(['a', 'a'])=False, expected True ✗ │      │
│             │                                                │         │ all_same([1, 1, 2])=False ✓                 │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "set("      │ "set(1 for x in lst) == {1} else False"        │ FAILED  │ all_same([1])=True ✓                        │      │
│             │                                                │         │ all_same([1, 1, 1])=True ✓                  │      │
│             │                                                │         │ all_same([1, 2])=True, expected False ✗     │      │
│             │                                                │         │ all_same(['a', 'a'])=True ✓                 │      │
│             │                                                │         │ all_same([1, 1, 2])=True, expected False ✗  │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "all"       │ "all(x == lst[0] for x in lst) else False"     │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "all("      │ "all(elt == lst[0] for elt in lst) else False" │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "lst[0]"    │ "lst[0] == lst[-1] else False"                 │ PASSED  │                                             │      │
├─────────────┼────────────────────────────────────────────────┼─────────┼─────────────────────────────────────────────┼──────┤
│ "lst.count" │ "lst.count(lst[0]) == len(lst) else False"     │ PASSED  │                                             │      │
└─────────────┴────────────────────────────────────────────────┴─────────┴─────────────────────────────────────────────┴──────┘

Summary: 7/9 passed, 2/9 failed

2026-01-23 17:43:46 __main__ INFO     Saved evaluation results to /disk/u/arnab/Codes/Projects/goals/results/Qwen3-Coder-30B-A3B-Instruct/eval_result_13.json


100%|██████████| 14/14 [14:04<00:00, 60.29s/it]


In [ ]:
# with open(os.path.join(save_dir, "eval_result_3.json"), "r") as f:
#     eval_demo = json.load(f)

# pretty_print_eval_results(eval_demo, output_file="results.txt")